# Model Quantization — `poultry-seg-large.pt`

Export the segmentation model to FP16 ONNX for GPU inference. Smaller, faster, good enough accuracy.

**Steps:**
1. Baseline size & speed
2. FP16 ONNX export (GPU) — **~2× smaller**
3. Resolution comparison — drop to 416/480 for more speed
4. Copy best model to `deploy/`

In [11]:
import sys, os, time, platform
from pathlib import Path

# Add project root so we use the bundled ultralytics fork
PROJECT_ROOT = Path().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from ultralytics import YOLO

MODEL_PATH = PROJECT_ROOT / "models" / "poultry-seg-large.pt"
EXPORT_DIR = PROJECT_ROOT / "models" / "quantized"
EXPORT_DIR.mkdir(exist_ok=True)

# Use sample images for calibration / test inference
SAMPLE_DIR = PROJECT_ROOT / "sampleimages"
SAMPLE_VIDEOS = PROJECT_ROOT / "samplevideos"

print(f"Project root:  {PROJECT_ROOT}")
print(f"Model path:    {MODEL_PATH}")
print(f"Model exists:  {MODEL_PATH.exists()}")
print(f"Export dir:     {EXPORT_DIR}")
print(f"Platform:      {platform.machine()}")
print(f"Python:        {sys.version.split()[0]}")

Project root:  C:\Users\ahmed\Development\poultry-vision
Model path:    C:\Users\ahmed\Development\poultry-vision\models\poultry-seg-large.pt
Model exists:  True
Export dir:     C:\Users\ahmed\Development\poultry-vision\models\quantized
Platform:      AMD64
Python:        3.12.9


## 1. Baseline — Original PyTorch Model

Measure the current model size, parameter count, and inference speed.

In [12]:
import torch
import numpy as np

model = YOLO(str(MODEL_PATH))

# Check GPU
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print(f"Device: {'GPU — ' + torch.cuda.get_device_name(0) if DEVICE == '0' else 'CPU'}")

file_size_mb = MODEL_PATH.stat().st_size / (1024 * 1024)
n_params_m = sum(p.numel() for p in model.model.parameters()) / 1e6

print(f"Model:       {MODEL_PATH.name}")
print(f"Size:        {file_size_mb:.1f} MB")
print(f"Parameters:  {n_params_m:.1f}M")
print(f"Classes:     {model.names}")

# Benchmark on GPU
dummy = np.random.randint(0, 255, (720, 1280, 3), dtype=np.uint8)
IMG_SIZE = 640

for _ in range(5):
    model.predict(dummy, imgsz=IMG_SIZE, device=DEVICE, verbose=False)

times = []
for _ in range(20):
    t0 = time.perf_counter()
    model.predict(dummy, imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    times.append((time.perf_counter() - t0) * 1000)

baseline_ms = np.median(times)
print(f"\nBaseline ({DEVICE}, {IMG_SIZE}px): {baseline_ms:.0f} ms, {1000/baseline_ms:.1f} FPS")

Device: GPU — NVIDIA GeForce RTX 5090
Model:       poultry-seg-large.pt
Size:        56.6 MB
Parameters:  29.2M
Classes:     {0: 'feeder', 1: 'hen', 2: 'waterer'}

Baseline (0, 640px): 17 ms, 58.2 FPS


## 2. Export FP16 ONNX (half precision, GPU)

FP16 halves the weight storage. ONNX with `half=True` requires GPU for export. This gives ~2× smaller file and faster GPU inference.

In [13]:
# Export FP16 ONNX — needs GPU
onnx_fp16_path = model.export(
    format="onnx",
    imgsz=IMG_SIZE,
    half=True,
    opset=18,
    simplify=False,
    device=DEVICE,
)
onnx_fp16_path = Path(onnx_fp16_path)
onnx_fp16_mb = onnx_fp16_path.stat().st_size / (1024 * 1024)

print(f"FP16 ONNX:   {onnx_fp16_path.name}")
print(f"Size:        {onnx_fp16_mb:.1f} MB  (was {file_size_mb:.1f} MB .pt)")
print(f"Reduction:   {(1 - onnx_fp16_mb / (n_params_m * 4)):.0%} vs raw FP32 weights")

Ultralytics 8.3.63  Python-3.12.9 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5090, 32607MiB)

PyTorch: starting from 'C:\Users\ahmed\Development\poultry-vision\models\poultry-seg-large.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) ((1, 39, 8400), (1, 32, 160, 160)) (56.6 MB)

ONNX: starting export with onnx 1.20.0 opset 18...
Applied 17 of general pattern rewrite rules.
ONNX: export success  5.6s, saved as 'C:\Users\ahmed\Development\poultry-vision\models\poultry-seg-large.onnx' (57.6 MB)

Export complete (5.8s)
Results saved to C:\Users\ahmed\Development\poultry-vision\models
Predict:         yolo predict task=segment model=C:\Users\ahmed\Development\poultry-vision\models\poultry-seg-large.onnx imgsz=640 half 
Validate:        yolo val task=segment model=C:\Users\ahmed\Development\poultry-vision\models\poultry-seg-large.onnx imgsz=640 data=/content/Poultry-Behaviour-Detection-7/data.yaml half 
Visualize:       https://netron.app
FP16 ONNX:   poultry-seg-large.onnx
S

In [14]:
# Benchmark FP16 ONNX on GPU
onnx_model = YOLO(str(onnx_fp16_path), task="segment")

for _ in range(5):
    onnx_model.predict(dummy, imgsz=IMG_SIZE, device=DEVICE, verbose=False)

onnx_times = []
for _ in range(20):
    t0 = time.perf_counter()
    onnx_model.predict(dummy, imgsz=IMG_SIZE, device=DEVICE, verbose=False)
    onnx_times.append((time.perf_counter() - t0) * 1000)

onnx_ms = np.median(onnx_times)
print(f"FP16 ONNX ({DEVICE}, {IMG_SIZE}px): {onnx_ms:.0f} ms, {1000/onnx_ms:.1f} FPS")
print(f"vs baseline: {baseline_ms/onnx_ms:.2f}x speedup")

Loading C:\Users\ahmed\Development\poultry-vision\models\poultry-seg-large.onnx for ONNX Runtime inference...
Using ONNX Runtime CUDAExecutionProvider


RuntimeError: Error when binding input: There's no data transfer registered for copying tensors from Device:[DeviceType:1 MemoryType:0 VendorId:4318 DeviceId:0 Alignment:0] to Device:[DeviceType:0 MemoryType:0 VendorId:0 DeviceId:0 Alignment:0]

## 3. Resolution Comparison

Lowering input resolution is the easiest way to get more speed. 640→416 gives ~2.5× speedup.

In [ ]:
# Speed at different resolutions using the FP16 ONNX model
print(f"{'Resolution':<12} {'Time (ms)':>10} {'FPS':>8} {'Speedup':>10}")
print("-" * 44)

ref_ms = None
for imgsz in [640, 480, 416, 320]:
    onnx_model.predict(dummy, imgsz=imgsz, device=DEVICE, verbose=False)
    t = []
    for _ in range(15):
        t0 = time.perf_counter()
        onnx_model.predict(dummy, imgsz=imgsz, device=DEVICE, verbose=False)
        t.append((time.perf_counter() - t0) * 1000)
    med = np.median(t)
    if ref_ms is None:
        ref_ms = med
    print(f"{imgsz}px{'':<6} {med:>10.0f} {1000/med:>8.1f} {ref_ms/med:>9.1f}x")

NotSupportedError: Compiled functions can't take variable number of arguments or use keyword-only arguments with defaults:
  File "C:\Users\ahmed\Development\poultry-vision\ultralytics\nn\tasks.py", line 95
    def forward(self, x, *args, **kwargs):
                                 ~~~~~~~ <--- HERE
        """
        Perform forward pass of the model for either training or inference.


## 4. Copy to Deploy

Copy the FP16 ONNX model to the `deploy/` folder and update the config.

In [ ]:
import shutil

deploy_dir = PROJECT_ROOT / "deploy"
dest = deploy_dir / onnx_fp16_path.name
shutil.copy2(onnx_fp16_path, dest)

print(f"Copied {onnx_fp16_path.name} → deploy/")
print(f"Size: {onnx_fp16_mb:.1f} MB")
print()
print("Update deploy/config.yaml:")
print(f'  model.path: "{onnx_fp16_path.name}"')
print(f'  model.device: "0"   # GPU')